# G1 Academy Bonus - Task 3: Gestures and Movement Commands


## Introduction
Extend Task 2's `chat_turn_with_knowledge` so the assistant can act, not just talk:
- **Announce its intention first** (e.g. say "Waving now" before waving) -- always tell the person what you are about to do before doing it.
- **Play a high-level gesture** when asked by name: `g1.face_wave()`, `g1.clap()`, `g1.high_wave()`, `g1.shake_hand()`, `g1.hug()`, `g1.left_kiss()` (no arguments, blocking, no HL gesture is expected to be interrupted mid-play).
- **Take small, fixed movement commands** with `g1.loco_move(vx, vy, vyaw, duration_s)`:

| Command | Call |
|---|---|
| step forward | `g1.loco_move(0.15, 0, 0, 1.0)` |
| step backward | `g1.loco_move(-0.15, 0, 0, 1.0)` |
| step left | `g1.loco_move(0, 0.15, 0, 1.0)` |
| step right | `g1.loco_move(0, -0.15, 0, 1.0)` |
| turn left | `g1.loco_move(0, 0, 0.5, 1.0)` |
| turn right | `g1.loco_move(0, 0, -0.5, 1.0)` |

**Safety:** call `g1.walk_mode(waist_locked=False)` in its own cell once before any movement command, and only run movement commands in a clear space -- these are small, fixed steps on purpose, do not make them bigger.

This notebook sets up its own `g1`, `client`, `history`, `knowledge`, and `chat_turn_with_knowledge` (you do not need to have run Tasks 1-2 first).


In [ ]:
import sys
sys.path.append("..")
from sdk_wrapper import G1
import ollama_api_util

g1 = G1(iface="eth0", domain_id=0)
client = ollama_api_util.get_client()
history = []

knowledge = ollama_api_util.load_knowledge_base("g1_academy_knowledge_de.json")

def chat_turn_with_knowledge(text):
    matches = ollama_api_util.retrieve_relevant(knowledge, text, top_k=3)
    if matches:
        context = "\n".join(f"F: {m['question']}\nA: {m['answer']}" for m in matches)
        augmented = f"Known facts:\n{context}\n\nQuestion: {text}"
    else:
        augmented = text  # nothing relevant found -- ask it plainly, don't force in noise
    reply = ollama_api_util.chat_reply(client, augmented, history=history)
    history.append({"role": "user", "content": text})
    history.append({"role": "assistant", "content": reply})
    print(reply)
    g1.say(reply, language="EN")
    return reply


## Codex prompt
```
I already have chat_turn_with_knowledge(text) from Task 2. I also have a G1 instance `g1` with these
no-argument high-level gestures: g1.face_wave(), g1.clap(), g1.high_wave(), g1.shake_hand(), g1.hug(),
g1.left_kiss(); and g1.loco_move(vx, vy, vyaw, duration_s) for movement, where a "step" is 1.0s at
vx=+-0.15 (forward/back), vy=+-0.15 (left/right), and a "turn" is 1.0s at vyaw=+-0.5 (left/right).

Write a new function chat_turn_with_actions(text) that:
1. First asks the model (via chat_turn_with_knowledge or a similar chat_reply call) to classify the
   user's message as either a plain question, a gesture request (and which gesture), or a movement
   command (and which direction) -- ask for a short structured answer you can parse, e.g. one word.
2. If it is a gesture or movement request: first call g1.say(...) announcing what you are about to do
   (in English), THEN call the matching g1 method from the tables above.
3. Otherwise, fall back to chat_turn_with_knowledge(text) for a normal spoken answer.

Keep everything else (history, printing, g1.say) consistent with chat_turn_with_knowledge.
```


In [ ]:
GESTURES = {
    "wave": g1.face_wave, "high wave": g1.high_wave, "clap": g1.clap,
    "shake hand": g1.shake_hand, "hug": g1.hug, "kiss": g1.left_kiss,
}
MOVES = {
    "forward": (0.15, 0, 0), "backward": (-0.15, 0, 0),
    "left": (0, 0.15, 0), "right": (0, -0.15, 0),
    "turn left": (0, 0, 0.5), "turn right": (0, 0, -0.5),
}


In [ ]:
# Run this only after the area is clear, before issuing movement commands.
g1.walk_mode(waist_locked=False)

In [ ]:
def chat_turn_with_actions(text):
    # Classified by keyword match rather than a second LLM call -- more reliable than
    # parsing a small model's free-form classification, and avoids doubling latency.
    # Longest names checked first so "turn left"/"turn right" win over plain "left"/"right".
    lowered = text.lower()
    gesture_name = next((name for name in sorted(GESTURES, key=len, reverse=True) if name in lowered), None)
    if gesture_name:
        reply = f"Okay, {gesture_name} now."
        g1.say(reply, language="EN")
        GESTURES[gesture_name]()
        history.append({"role": "user", "content": text})
        history.append({"role": "assistant", "content": reply})
        print(reply)
        return reply
    move_name = next((name for name in sorted(MOVES, key=len, reverse=True) if name in lowered), None)
    if move_name:
        reply = f"Okay, moving {move_name} now."
        g1.say(reply, language="EN")
        g1.loco_move(*MOVES[move_name], duration_s=1.0)
        history.append({"role": "user", "content": text})
        history.append({"role": "assistant", "content": reply})
        print(reply)
        return reply
    return chat_turn_with_knowledge(text)


In [ ]:
chat_turn_with_actions(input("Du: "))
